# Ders 08 - Çoklu Temsilci Tasarım Deseni


## Kurulum


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Neden Çoklu Ajan Sistemleri?

Gerçek dünya görevleri, seyahat planlaması gibi, lojistik, yerel bilgi, bütçeleme ve daha fazlası gibi birçok farklı uzmanlık türünü içerir. Her şeyi tek başına halletmeye çalışan bir ajan hızla yönetilemez hale gelir.

Çoklu ajan sistemleri bunu **uzmanlaşma** yoluyla çözer: her ajan bir uzmanlık alanına odaklanır ve bir genelciye kıyasla daha yüksek kaliteli sonuçlar üretir. Ayrıca **ölçeklenebilirliği** artırır — mevcut iş akışını yeniden yazmadan yeni ajanlar (örneğin, bir uçuş uzmanı, bir restoran eleştirmeni) ekleyebilirsiniz. Ajanlar, bağlamı birinden diğerine aktaran yapılandırılmış bir boru hattı aracılığıyla bir araya gelir.


## Uzman Ajanlar Yaratma


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Ardışık Bir İş Akışı Oluşturma

`WorkflowBuilder`, ajanları yönlendirilmiş bir grafik halinde birbirine bağlamanızı sağlar. Burada basit iki adımlı bir boru hattı oluşturuyoruz: **TravelPlanner** seyahat planını taslak olarak hazırlar, ardından **TravelConcierge** inceleyip geliştirir.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## İş Akışına Daha Fazla Ajan Ekleme

Çok ajanlı desenin en büyük avantajlarından biri genişletmenin ne kadar kolay olduğudur. Aşağıda bütçeyi seyahatçinin bütçesiyle karşılaştıran, masrafları sınırlamanın üzerine çıkabilecek öğeleri işaretleyen ve para tasarrufu sağlayan alternatifler öneren **BudgetReviewer** ajanını ekliyoruz. İş akışı şimdi üç ajanı ardışık olarak çalıştırıyor:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Özet

Bu derste şunları öğrendiniz:

1. **Uzmanlaşmış ajanlar oluşturma** — her biri belirli bir role (planlama, konsiyerj, bütçe incelemesi) sahip.
2. **Ajanları sıralı bir iş akışına bağlama** `WorkflowBuilder` ve `add_edge` kullanarak.
3. **Çok ajanlı bir boru hattından çıktı akışı sağlama**, hangi ajanın konuştuğunu takip ederek.
4. **Bir iş akışını genişletme**, zincire yeni ajanlar ekleyerek mevcut ajanları değiştirmeden.

Çok ajanlı tasarım deseni, her ajanı basit tutarken, tek bir ajanın tek başına elde edebileceğinden daha zengin ve daha detaylı incelenmiş sonuçlar üretir.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, yapay zeka çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstermemize rağmen, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi diliyle yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu oluşabilecek yanlış anlamalar veya yorum hatalarından sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
